In [ ]:
# ==========================================
# 1. SETUP & LOGIN
# ==========================================
!pip install -q --upgrade pip
!pip install -q --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio wandb huggingface_hub audiomentations soundfile librosa

import os
import torch
from huggingface_hub import login
import wandb

# 1. Login to WandB
wandb.login(key="wandb_key_here")

# 2. Login to Hugging Face
write_token = "HF_token_here"
login(token=write_token)

# Define Model & Repo
repo_name = "whisper-small-n-demo-final-augmented-new-data-X"
username = "wandererupak"
repo_id = f"{username}/{repo_name}"
model_id = 'openai/whisper-small'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rupaktiwari (rupaktiwari-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
# ==========================================
# 2. DATA AUGMENTATION & PREP
# ==========================================
from datasets import load_dataset, Audio
from transformers import WhisperTokenizer, WhisperProcessor, WhisperFeatureExtractor
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
import numpy as np

print("⬇️ Loading Newari dataset (n-demo-ultimate)...")
dataset = load_dataset("wandererupak/n-demo-ultimate")

# 1. Standardize columns
for split in dataset.keys():
    if 'transcription' in dataset[split].column_names:
        dataset[split] = dataset[split].rename_column("transcription", "sentence")
    if 'utterance_id' in dataset[split].column_names:
        dataset[split] = dataset[split].rename_column("utterance_id", "id")

# 2. Load Processors
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language='Nepali', task='transcribe')
processor = WhisperProcessor.from_pretrained(model_id, language='Nepali', task='transcribe')

# 3. Cast Audio (Lazy Loading)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# --- DEFINE AUGMENTATION ---
print("🛠️ Setting up Augmentation Pipeline...")
augmentations = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
    PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
])

# Function 1: Augment + Process (For TRAIN split only)
def augment_dataset(batch):
    input_features_list = []
    labels_list = []

    for audio_dict, sentence in zip(batch["audio"], batch["sentence"]):
        audio_array = audio_dict["array"]
        sr = audio_dict["sampling_rate"]

        # Apply Augmentation (Try/Except to be safe)
        try:
            augmented_audio = augmentations(samples=audio_array, sample_rate=sr)
        except:
            augmented_audio = audio_array # Fallback if augmentation fails

        # Feature Extraction
        features = feature_extractor(augmented_audio, sampling_rate=sr).input_features[0]
        input_features_list.append(features)

        # Tokenization
        label_ids = tokenizer(sentence).input_ids
        labels_list.append(label_ids)

    return {"input_features": input_features_list, "labels": labels_list}

# Function 2: Just Process (For VALIDATION & TEST splits - No Noise!)
def prepare_dataset_clean(batch):
    input_features_list = []
    labels_list = []

    for audio_dict, sentence in zip(batch["audio"], batch["sentence"]):
        audio_array = audio_dict["array"]
        sr = audio_dict["sampling_rate"]

        # Just extract features (No augmentation)
        features = feature_extractor(audio_array, sampling_rate=sr).input_features[0]
        input_features_list.append(features)

        label_ids = tokenizer(sentence).input_ids
        labels_list.append(label_ids)

    return {"input_features": input_features_list, "labels": labels_list}

print("⚙️ Applying transforms (Lazy Evaluation)...")
# This runs INSTANTLY because it doesn't process data yet. It waits for training.
dataset["train"].set_transform(augment_dataset)
dataset["validation"].set_transform(prepare_dataset_clean)
dataset["test"].set_transform(prepare_dataset_clean)

print("✅ Data Ready!")

⬇️ Loading Newari dataset (n-demo-ultimate)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/615 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/432M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

🛠️ Setting up Augmentation Pipeline...
⚙️ Applying transforms (Lazy Evaluation)...
✅ Data Ready!


In [3]:
# ==========================================
# 3. COLLATOR & METRICS
# ==========================================
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import jiwer

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * jiwer.wer(label_str, pred_str)
    cer = 100 * jiwer.cer(label_str, pred_str)

    return {"wer": wer, "cer": cer}

In [4]:
# ==========================================
# 4. TRAINING CONFIG (FIXED)
# ==========================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, set_seed
set_seed(42)
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Force language and task
model.generation_config.language = "nepali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.dropout = 0.2
model.config.attention_dropout = 0.2

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# --- DYNAMIC STEP CALCULATION ---
batch_size = 8
grad_accum = 2
steps_per_epoch = len(dataset["train"]) // (batch_size * grad_accum)
save_steps_count = int(steps_per_epoch / 1)
if save_steps_count < 1: save_steps_count = 1

print(f"💾 Saving model every {save_steps_count} steps.")

# Initialize WandB
wandb.init(project="whisper-small-final-augmented-new-data-X", name="n-final-aug-public-new-data-X", resume="allow")

training_args = Seq2SeqTrainingArguments(
    output_dir=repo_name,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=1e-5,
    warmup_ratio=0.1,  # You can keep this, the warning is harmless
    num_train_epochs=15,
    lr_scheduler_type="cosine",
    fp16=True,

    # --- LOGGING & SAVING ---
    logging_strategy="steps",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=save_steps_count,
    save_strategy="steps",
    save_steps=save_steps_count,
    save_total_limit=2,

    # --- DATA LOADING ---
    remove_unused_columns=False, # Required for augmentation
    dataloader_num_workers=2,    # CPU workers for augmentation

    # --- HUB SETTINGS ---
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_private_repo=False,
    hub_strategy="checkpoint",

    # --- GENERATION ---
    predict_with_generate=True,
    generation_max_length=225,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,

    # --- THE FIX IS HERE ---
    # We pass 'processor' because it contains BOTH the tokenizer (text) and feature extractor (audio)
    processing_class=processor,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)]
)

print("🚀 Starting Augmented Training...")
trainer.train()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

💾 Saving model every 285 steps.


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=4290, training_loss=0.4290679066192298, metrics={'train_runtime': 11848.762, 'train_samples_per_second': 5.792, 'train_steps_per_second': 0.362, 'total_flos': 1.980417309696e+19, 'train_loss': 0.4290679066192298, 'epoch': 15.0})

In [5]:
# ==========================================
# 5. TEST EVALUATION & FINAL PUSH
# ==========================================

print("\n🧪 Running evaluation on Test set (Clean Data)...")
# The test set automatically uses the 'prepare_dataset_clean' function
test_metrics = trainer.predict(dataset["test"]).metrics
print(f"🏆 Final Test Metrics: {test_metrics}")

print("\n☁️ Pushing final augmented model to Hugging Face (PUBLIC)...")
kwargs = {
    "dataset_tags": "wandererupak/n-demo-ultimate-Aug",
    "dataset": "N Demo Final (Augmented)",
    "language": "ne",
    "model_name": "Whisper Small N- Augmented",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)
processor.push_to_hub(repo_id)
print(f"✅ DONE! Live at: https://huggingface.co/{repo_id}")


🧪 Running evaluation on Test set (Clean Data)...


🏆 Final Test Metrics: {'test_loss': 0.5006421804428101, 'test_wer': 48.80560131795716, 'test_cer': 17.888675623800385, 'test_runtime': 155.7104, 'test_samples_per_second': 3.699, 'test_steps_per_second': 0.462}

☁️ Pushing final augmented model to Hugging Face (PUBLIC)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-data-X/training_args.bin: 100%|##########| 5.46kB / 5.46kB            

  ...-data-X/model.safetensors:   3%|3         | 33.5MB /  967MB            

README.md: 0.00B [00:00, ?B/s]

✅ DONE! Live at: https://huggingface.co/wandererupak/whisper-small-n-demo-final-augmented-new-data-X
